# 02. Data Preprocessing and Cohort Subsampling
**CSE437 Final Project: Disparity and Error Analysis in Linear Income Classification**  
*Texas 2023 ACS 1-Year PUMS Microdata*

---

### Purpose and Methodological Justifications
1. **Type Sanitization & Missing Value Handling:** Coerce Census sentinels and whitespace strings to numeric, dropping missing records[cite: 7]. Dropping structurally unrecorded individuals is methodologically justified because the classification task is defined strictly over the active labor force.
2. **Population Filtering:** Restrict to civilian workers actively employed full-time (`ESR == 1`, `WKHP >= 35`, `PERNP > 0`)[cite: 7].
3. **Outlier Controls:**
   - **Age Truncation (16–80):** Mitigates high-variance sparse tails in extreme senior labor (>80) without distorting core career trajectories[cite: 7].
   - **Weekly Hours Clipping (`WKHP <= 98`):** Winsorizes extreme reporting artifacts at 98 hours to prevent undue leverage in linear optimization while keeping valid records[cite: 7].
4. **Domain Feature Aggregation:**
   - **Occupational Bucketing:** Collapses ~500 granular 4-digit SOC codes (`OCCP`) into 12 broad functional domains based on federal labor taxonomies, reducing categorical dimensionality by 97.6% to avoid extreme sparsity in one-hot matrices[cite: 7].
   - **Class of Worker (`COW`):** Decodes integer codes into 8 institutional sectors[cite: 7].
5. **Target Construction & Sampling:** Computes the empirical 75th percentile cutoff ($90,000.00) and extracts a reproducible 30,000-record sample (`random_state=42`) for stable linear classifier convergence[cite: 7].

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Portable path resolution relative to repository root
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DATA_PATH = REPO_ROOT / "data" / "raw" / "psam_p48.csv"
OUTPUT_DIR = REPO_ROOT / "data" / "processed"
OUTPUT_FILE = OUTPUT_DIR / "texas_cleaned_30k.csv"

# Fallback resolution
if not RAW_DATA_PATH.exists():
    if (Path.cwd() / "psam_p48.csv").exists():
        RAW_DATA_PATH = Path.cwd() / "psam_p48.csv"
    elif (REPO_ROOT / "psam_p48.csv").exists():
        RAW_DATA_PATH = REPO_ROOT / "psam_p48.csv"
    else:
        raise FileNotFoundError(
            f"Could not locate 'psam_p48.csv'. Looked in:\n"
            f" - {RAW_DATA_PATH}\n"
            f" - {Path.cwd() / 'psam_p48.csv'}\n"
            "Please ensure raw microdata is in data/raw/ or run python src/download_data.py"
        )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[STATUS] Project root: {REPO_ROOT.resolve()}")
print(f"[STATUS] Loading raw microdata from: {RAW_DATA_PATH.resolve()}")

## 1. Variable Loading and Type Sanitization
We load the 10 candidate demographic, economic, and occupational variables, coerce non-numeric sentinels to `NaN`, and drop missing values across the active labor universe[cite: 7].

In [ ]:
selected_cols = [
    "PERNP",  # Total person's annual earnings[cite: 7]
    "WKHP",  # Usual weekly hours worked[cite: 7]
    "ESR",  # Employment status recode[cite: 7]
    "AGEP",  # Age[cite: 7]
    "SEX",  # Sex identifier[cite: 7]
    "SCHL",  # Educational attainment level[cite: 7]
    "MAR",  # Marital status[cite: 7]
    "COW",  # Class of worker / institutional sector[cite: 7]
    "OCCP",  # 4-digit SOC occupation code[cite: 7]
    "WKWN",  # Weeks worked past 12 months[cite: 7]
]

print(f"[STATUS] Ingesting {len(selected_cols)} core attributes...")
df_raw = pd.read_csv(RAW_DATA_PATH, usecols=selected_cols, low_memory=False)
initial_shape = df_raw.shape
print(f"Raw Ingested Records: {initial_shape[0]:,} rows, {initial_shape[1]} columns")

# Coerce non-numeric sentinels and drop missing entries[cite: 7]
for col in selected_cols:
    df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")

df_clean = df_raw.dropna(subset=selected_cols).copy()
print(f"Records After Dropping Structural Nulls: {len(df_clean):,} (Dropped {initial_shape[0] - len(df_clean):,} rows)")

## 2. Cohort Filtering and Outlier Management
We restrict records to active full-time workers (`ESR == 1`, `WKHP >= 35`, `PERNP > 0`), bound the age span to 16–80, and clip weekly hours at 98[cite: 7].

In [ ]:
# 1. Full-time civilian labor criteria[cite: 7]
df_workforce = df_clean[
    (df_clean["ESR"] == 1) & 
    (df_clean["WKHP"] >= 35) & 
    (df_clean["PERNP"] > 0)
].copy()
print(f"Workforce Filter Applied: {len(df_workforce):,} civilian full-time records retained.")

# 2. Senior tail outlier control (Ages 16-80)[cite: 7]
df_workforce = df_workforce[(df_workforce["AGEP"] >= 16) & (df_workforce["AGEP"] <= 80)].copy()

# 3. Work hours outlier winsorization (Cap WKHP at 98)[cite: 7]
hours_clipped_count = (df_workforce["WKHP"] > 98).sum()
df_workforce["WKHP"] = df_workforce["WKHP"].clip(upper=98)

print(f"Age Bounded (16-80):      {len(df_workforce):,} records remaining.")
print(f"Hours Worked Outliers:    {hours_clipped_count} records clipped at 98 hours.")

## 3. Domain Feature Aggregation
Collapsing ~500 granular 4-digit SOC codes into 12 broad occupational groups prevents high-cardinality overfitting while preserving sector dynamics[cite: 7]. We also map integer `COW` codes to readable sector names[cite: 7].

In [ ]:
def map_occupation_bucket(code: float) -> str:
    """
    Maps ~500 granular 4-digit Census OCCP codes to 12 broad SOC domains[cite: 7].
    """
    c = int(code)
    if 10 <= c <= 960:
        return "Management_Business_Finance"
    elif 1005 <= c <= 1980:
        return "STEM_Science_Architecture"
    elif 2001 <= c <= 2555:
        return "Education_Legal_Social"
    elif 2600 <= c <= 2970:
        return "Arts_Media_Design"
    elif 3000 <= c <= 3655:
        return "Healthcare"
    elif 3700 <= c <= 3960:
        return "Protective_Service"
    elif 4000 <= c <= 4655:
        return "Service_Food_Personal_Cleaning"
    elif 4700 <= c <= 5940:
        return "Sales_Office_Admin"
    elif 6005 <= c <= 6950:
        return "Construction_Extraction_Farming"
    elif 7000 <= c <= 7640:
        return "Installation_Maintenance_Repair"
    elif 7700 <= c <= 8990:
        return "Production_Manufacturing"
    elif 9005 <= c <= 9760:
        return "Transportation_Material_Moving"
    return "Other"

# Apply occupational mapping[cite: 7]
df_workforce["OCCP_GROUP"] = df_workforce["OCCP"].apply(map_occupation_bucket)

# Map Class of Worker (COW) sectors[cite: 7]
cow_labels = {
    1: "Private_ForProfit",
    2: "Private_NonProfit",
    3: "Local_Gov",
    4: "State_Gov",
    5: "Federal_Gov",
    6: "Self_Employed_NotInc",
    7: "Self_Employed_Inc",
    8: "Without_Pay",
}
df_workforce["COW_GROUP"] = df_workforce["COW"].map(cow_labels).fillna("Other")

print("Occupational Distribution Across 12 Domains:")
print(df_workforce["OCCP_GROUP"].value_counts())
print("\nEmployment Sector Breakdown:")
print(df_workforce["COW_GROUP"].value_counts())

## 4. Target Threshold Calculation and 30,000 Cohort Subsampling
We set the top-quartile income cutoff at the empirical 75th percentile ($90,000.00) on the preprocessed workforce and extract a 30,000-record subset with a fixed seed (`random_state=42`)[cite: 7].

In [ ]:
# Compute empirical 75th percentile threshold[cite: 7]
p75_threshold = df_workforce["PERNP"].quantile(0.75)
df_workforce["HIGH_EARNER"] = (df_workforce["PERNP"] >= p75_threshold).astype(int)

print("=" * 65)
print(f"Top-Quartile Target Cutoff (P75): ${p75_threshold:,.2f}")
print("Target Class Breakdown on Full Cohort:")
print(df_workforce["HIGH_EARNER"].value_counts(normalize=True).apply(lambda x: f"{x*100:.2f}%"))
print("=" * 65)

# Subsample 30,000 records for efficient, stable LinearSVC convergence[cite: 7]
df_sampled_30k = df_workforce.sample(n=30000, random_state=42).reset_index(drop=True)

print("Target Class Breakdown in 30,000 Sample:")
sample_balance = df_sampled_30k["HIGH_EARNER"].value_counts(normalize=True) * 100
print(f"  Class 0 (Standard Earner): {sample_balance[0]:.2f}%")
print(f"  Class 1 (High Earner):     {sample_balance[1]:.2f}%")

## 5. Pipeline Verification Table & Artifact Export
Validating stage shapes against Report Table 2.5 and exporting `data/processed/texas_cleaned_30k.csv`[cite: 5].

In [ ]:
# Export cleaned 30k sample dataset[cite: 7]
df_sampled_30k.to_csv(OUTPUT_FILE, index=False)

print("=" * 65)
print(f"[SUCCESS] Cleaned dataset successfully exported to:\n -> {OUTPUT_FILE.resolve()}")
print(f"Final Export Shape: {df_sampled_30k.shape[0]:,} rows, {df_sampled_30k.shape[1]} columns")
print("=" * 65)

# Report Table 2.5 Pipeline Traceability Audit
pipeline_audit = pd.DataFrame([
    {"Stage": "Raw Microdata", "Rows": initial_shape[0], "Columns": 288, "Notes": "Unfiltered Texas PUMS survey records"},
    {"Stage": "Variable Selection", "Rows": initial_shape[0], "Columns": len(selected_cols), "Notes": "Retained 10 core fields"},
    {"Stage": "Structural Nulls Dropped", "Rows": len(df_clean), "Columns": len(selected_cols), "Notes": "Removed non-labor/missing entries"},
    {"Stage": "Active Labor & Outlier Truncation", "Rows": len(df_workforce), "Columns": len(selected_cols), "Notes": "ESR=1, WKHP>=35, PERNP>0, Age 16-80, WKHP<=98[cite: 7]"},
    {"Stage": "Domain Engineering", "Rows": len(df_workforce), "Columns": df_workforce.shape[1], "Notes": "Engineered OCCP_GROUP, COW_GROUP, HIGH_EARNER[cite: 7]"},
    {"Stage": "Cleaned Subsample Export", "Rows": df_sampled_30k.shape[0], "Columns": df_sampled_30k.shape[1], "Notes": "Final 30k sample written to data/processed/[cite: 7]"}
])

print("\nPipeline Stage Summary (Matches Report Section 2.5):")
print(pipeline_audit.to_string(index=False))